# RNN (Recurrent Neural Network) — Text Classification on AG News

This notebook demonstrates how to build a **vanilla (Elman) RNN** for **text classification** using **PyTorch**.

**What you will learn:**
- How a basic RNN processes sequences one token at a time
- The vanishing gradient problem and why vanilla RNNs struggle with long sequences
- How to build a simple text classifier from embeddings through to prediction
- How RNNs compare to LSTMs and GRUs

**Dataset:** AG News — 120,000 news articles in 4 classes:
- 0: World
- 1: Sports
- 2: Business
- 3: Sci/Tech

Loaded via HuggingFace `datasets` (auto-downloads).

## Step 1 — Install Required Packages

In [6]:
!pip install torch datasets --quiet

## Step 2 — Imports & Device Setup

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Step 3 — Load & Explore the AG News Dataset

AG News is a topic classification dataset. Each sample has a `text` field (headline + description) and a `label` (0-3).

In [8]:
raw_dataset = load_dataset("ag_news")

class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

print(f"Train samples: {len(raw_dataset['train'])}")
print(f"Test samples: {len(raw_dataset['test'])}")

# Show some examples
print("\n--- Sample Articles ---")
for i in range(4):
    sample = raw_dataset['train'][i]
    print(f"\n[{class_names[sample['label']]}] {sample['text'][:120]}...")

# Class distribution
print("\n--- Class Distribution (train) ---")
label_counts = Counter(raw_dataset['train']['label'])
for label, count in sorted(label_counts.items()):
    print(f"  {class_names[label]}: {count:,}")

Train samples: 120000
Test samples: 7600

--- Sample Articles ---

[Business] Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...

[Business] Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputat...

[Business] Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the ou...

[Business] Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the...

--- Class Distribution (train) ---
  World: 30,000
  Sports: 30,000
  Business: 30,000
  Sci/Tech: 30,000


## Step 4 — Tokenization & Vocabulary Building

We build a vocabulary from the training data:
1. Tokenize each article into words
2. Count word frequencies
3. Keep the top 25,000 most common words
4. Map each word to a unique integer index

In [9]:
def tokenize(text):
    """Simple whitespace tokenizer with lowercasing."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.split()

# Count word frequencies
word_counts = Counter()
for sample in raw_dataset['train']:
    word_counts.update(tokenize(sample['text']))

print(f"Unique words: {len(word_counts)}")

# Build vocabulary
VOCAB_SIZE = 25000
PAD_IDX = 0
UNK_IDX = 1

word_to_idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in word_counts.most_common(VOCAB_SIZE - 2):
    word_to_idx[word] = len(word_to_idx)

print(f"Vocabulary size: {len(word_to_idx)}")

Unique words: 102169
Vocabulary size: 25000


## Step 5 — Dataset & DataLoader

We create a simple Dataset that tokenizes, numericalizes, and truncates text. We use a collate function to pad sequences in each batch.

In [10]:
MAX_LEN = 128

class NewsDataset(Dataset):
    def __init__(self, hf_dataset, word_to_idx, max_len=128):
        self.texts = hf_dataset['text']
        self.labels = hf_dataset['label']
        self.word_to_idx = word_to_idx
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])[:self.max_len]
        indices = [self.word_to_idx.get(t, UNK_IDX) for t in tokens]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    max_len = max(len(seq) for seq in sequences)
    padded = torch.zeros(len(sequences), max_len, dtype=torch.long)
    for i, seq in enumerate(sequences):
        padded[i, :len(seq)] = seq
    return padded, torch.stack(labels), lengths

# Create datasets and loaders
BATCH_SIZE = 64

train_dataset = NewsDataset(raw_dataset['train'], word_to_idx, MAX_LEN)
test_dataset = NewsDataset(raw_dataset['test'], word_to_idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Inspect a batch
seqs, labels, lengths = next(iter(train_loader))
print(f"\nBatch shapes — Sequences: {seqs.shape}, Labels: {labels.shape}")

Train batches: 1875
Test batches: 119

Batch shapes — Sequences: torch.Size([64, 54]), Labels: torch.Size([64])


## Step 6 — Define the Vanilla RNN Model

Architecture:
```
Input (token indices)
  → Embedding(vocab_size, embed_dim)       — word → dense vector
  → RNN(embed_dim, hidden_dim, layers=1)   — process sequence step by step
  → Take last hidden state                 — sequence summary
  → Dropout(0.5)                           — regularization
  → Linear(hidden_dim, num_classes)         — classify into 4 categories
```

**How a vanilla RNN works:**

At each time step $t$, the RNN computes:

$$h_t = \tanh(W_{ih} \cdot x_t + W_{hh} \cdot h_{t-1} + b)$$

Where:
- $x_t$: input at time step $t$ (current word embedding)
- $h_{t-1}$: hidden state from previous step (memory)
- $W_{ih}$, $W_{hh}$: learnable weight matrices

**The vanishing gradient problem:**
- During backpropagation through time (BPTT), gradients are multiplied by $W_{hh}$ at each step
- If $||W_{hh}|| < 1$, gradients shrink exponentially → model forgets long-range dependencies
- If $||W_{hh}|| > 1$, gradients explode → training becomes unstable
- This is why LSTMs and GRUs were invented — they have gating mechanisms that allow gradients to flow

In [11]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super(VanillaRNN, self).__init__()
        
        # Embedding: token index → dense vector
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        
        # Vanilla RNN layer
        # nonlinearity='tanh' is the default — applies tanh to the hidden state
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,         # Single layer — keeps it simple
            batch_first=True,
            nonlinearity='tanh'
        )
        
        # Classification head
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x, lengths):
        # Embed
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        
        # Pack for efficient processing
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # Run RNN
        packed_output, h_n = self.rnn(packed)
        # h_n: (num_layers, batch, hidden_dim)
        
        # Take last hidden state
        hidden = h_n[-1]  # (batch, hidden_dim)
        
        # Classify
        hidden = self.dropout(hidden)
        output = self.fc(hidden)  # (batch, num_classes)
        return output

# Hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 4

model = VanillaRNN(
    vocab_size=len(word_to_idx),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    pad_idx=PAD_IDX
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

VanillaRNN(
  (embedding): Embedding(25000, 128, padding_idx=0)
  (rnn): RNN(128, 256, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)

Total parameters: 3,299,844


## Step 7 — Loss Function & Optimizer

- **CrossEntropyLoss**: For multi-class classification (4 classes). Combines LogSoftmax + NLLLoss.
- **Adam**: Adaptive learning rate optimizer.

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss: CrossEntropyLoss")
print("Optimizer: Adam (lr=0.001)")

Loss: CrossEntropyLoss
Optimizer: Adam (lr=0.001)


## Step 8 — Training Loop

We train for 3 epochs. Gradient clipping is important for RNNs to prevent exploding gradients.

In [13]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (sequences, labels, lengths) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward
        outputs = model(sequences, lengths)
        loss = criterion(outputs, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Clip gradients
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch [{epoch+1}/{num_epochs}], "
                  f"Batch [{batch_idx+1}/{len(train_loader)}], "
                  f"Loss: {loss.item():.4f}")
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] — Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

print("\nTraining complete!")

  Epoch [1/3], Batch [200/1875], Loss: 1.0567
  Epoch [1/3], Batch [400/1875], Loss: 1.0919
  Epoch [1/3], Batch [600/1875], Loss: 0.8516
  Epoch [1/3], Batch [800/1875], Loss: 0.9406
  Epoch [1/3], Batch [1000/1875], Loss: 0.8181
  Epoch [1/3], Batch [1200/1875], Loss: 0.8570
  Epoch [1/3], Batch [1400/1875], Loss: 0.6005
  Epoch [1/3], Batch [1600/1875], Loss: 0.5868
  Epoch [1/3], Batch [1800/1875], Loss: 0.7014
Epoch [1/3] — Loss: 0.8452, Accuracy: 65.73%
  Epoch [2/3], Batch [200/1875], Loss: 0.6274
  Epoch [2/3], Batch [400/1875], Loss: 0.4465
  Epoch [2/3], Batch [600/1875], Loss: 0.4614
  Epoch [2/3], Batch [800/1875], Loss: 0.4412
  Epoch [2/3], Batch [1000/1875], Loss: 0.1995
  Epoch [2/3], Batch [1200/1875], Loss: 0.7218
  Epoch [2/3], Batch [1400/1875], Loss: 0.4753
  Epoch [2/3], Batch [1600/1875], Loss: 0.2943
  Epoch [2/3], Batch [1800/1875], Loss: 0.4742
Epoch [2/3] — Loss: 0.4971, Accuracy: 83.79%
  Epoch [3/3], Batch [200/1875], Loss: 0.3173
  Epoch [3/3], Batch [400/

## Step 9 — Evaluate on Test Set

We compute overall accuracy and per-class accuracy to see which news categories the model handles best.

In [14]:
model.eval()
correct = 0
total = 0
class_correct = [0] * NUM_CLASSES
class_total = [0] * NUM_CLASSES

with torch.no_grad():
    for sequences, labels, lengths in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences, lengths)
        _, predicted = torch.max(outputs, 1)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        for i in range(labels.size(0)):
            label = labels[i].item()
            class_correct[label] += (predicted[i] == labels[i]).item()
            class_total[label] += 1

print(f"Overall Test Accuracy: {100 * correct / total:.2f}%")
print(f"\nPer-Class Accuracy:")
for i in range(NUM_CLASSES):
    acc = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
    print(f"  {class_names[i]:>10s}: {acc:.1f}%")

Overall Test Accuracy: 86.51%

Per-Class Accuracy:
       World: 84.3%
      Sports: 92.7%
    Business: 78.3%
    Sci/Tech: 90.7%


## Step 10 — Test on Custom Texts

In [15]:
def predict_news(text, model, word_to_idx):
    """Predict the category of a news article."""
    model.eval()
    tokens = tokenize(text)[:MAX_LEN]
    indices = [word_to_idx.get(t, UNK_IDX) for t in tokens]
    seq = torch.tensor([indices], dtype=torch.long).to(device)
    length = torch.tensor([len(indices)])
    
    with torch.no_grad():
        output = model(seq, length)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()
    
    return class_names[pred], confidence

test_articles = [
    "The president met with foreign leaders to discuss trade agreements and diplomacy.",
    "The team won the championship game with a last-second goal in overtime.",
    "Apple reported record quarterly revenue driven by strong iPhone sales.",
    "Researchers developed a new AI algorithm that can predict protein structures."
]

print("=== Custom News Predictions ===")
for article in test_articles:
    category, conf = predict_news(article, model, word_to_idx)
    print(f"\n\"{article[:80]}...\"")
    print(f"  → {category} (confidence: {conf:.4f})")

=== Custom News Predictions ===

"The president met with foreign leaders to discuss trade agreements and diplomacy..."
  → World (confidence: 0.9919)

"The team won the championship game with a last-second goal in overtime...."
  → Sports (confidence: 0.9924)

"Apple reported record quarterly revenue driven by strong iPhone sales...."
  → Sci/Tech (confidence: 0.6198)

"Researchers developed a new AI algorithm that can predict protein structures...."
  → Sci/Tech (confidence: 0.9427)


## RNN vs LSTM vs GRU — Comparison

| Feature | Vanilla RNN | LSTM | GRU |
|---------|------------|------|-----|
| **Hidden state** | Single $h_t$ | Two: $h_t$ (hidden) + $c_t$ (cell) | Single $h_t$ |
| **Gates** | None | 3 (forget, input, output) | 2 (reset, update) |
| **Long-range memory** | Poor (vanishing gradients) | Excellent (cell state) | Good (update gate) |
| **Parameters** | Fewest | Most | Medium |
| **Speed** | Fastest | Slowest | Medium |
| **Best for** | Short sequences | Long sequences, complex patterns | Good balance of speed & performance |

**Takeaway:** Vanilla RNNs are simple to understand but struggle with long sequences. For real-world tasks, use LSTM or GRU.

## Summary

In this notebook we:
1. **Loaded** the AG News dataset (120k articles, 4 classes)
2. **Built** a vocabulary and data pipeline
3. **Implemented** a vanilla RNN classifier
4. **Trained** for 3 epochs with gradient clipping
5. **Evaluated** overall and per-class accuracy
6. **Compared** RNN vs LSTM vs GRU architectures

**Key takeaways:**
- Vanilla RNNs are the simplest recurrent architecture
- They suffer from vanishing gradients for long sequences
- Gradient clipping is essential to prevent exploding gradients
- For longer or more complex sequences, prefer LSTM or GRU